# Milestone 4 — Vector Retrieval với FAISS
Notebook chạy embedding trên Colab T4, lưu FAISS index và metadata trên Google Drive. Không triển khai Hybrid Retrieval trong notebook này.

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Master/IT2043/Project/Final_Project')
PROJECT_ROOT = DRIVE_ROOT / 'vn-landlaw-qa'
REPO_URL = 'https://github.com/thinhGithub/vn-landlaw-qa.git'
print('Drive root:', DRIVE_ROOT)
print('Project root:', PROJECT_ROOT)

Drive root: /content/drive/MyDrive/Master/IT2043/Project/Final_Project
Project root: /content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa


## Đồng bộ private repository
Tạo Colab Secret tên `GITHUB_TOKEN`, bật quyền truy cập notebook. Cell không in token hoặc authorization header.

In [20]:
import os

os.environ["GITHUB_TOKEN"] = "github_pat_xxxxxxxxxxxxx"

In [ ]:
from google.colab import userdata
import base64
import subprocess

github_user = 'thinhGithub'
github_token = os.environ["GITHUB_TOKEN"]
assert github_token, 'Chưa cấu hình GITHUB_TOKEN trong Colab Secrets'
auth = base64.b64encode(f'{github_user}:{github_token}'.encode()).decode()
header = f'http.extraHeader=Authorization: Basic {auth}'

if (PROJECT_ROOT / '.git').exists():
    command = ['git', '-C', str(PROJECT_ROOT), '-c', header, 'pull', '--ff-only']
    action = 'pull'
else:
    command = ['git', '-c', header, 'clone', REPO_URL, str(PROJECT_ROOT)]
    action = 'clone'

sync_result = subprocess.run(command, check=True, capture_output=True, text=True)
print(f'Git {action} thành công')
del github_token, auth, header, command, sync_result

TimeoutException: Requesting secret GITHUB_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [18]:
%cd $PROJECT_ROOT
!git rev-parse --short HEAD
!git status --short

/content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa
597259d
?? "data/processed/Lua\314\243\314\202t-31-2024-QH15.stats.json"
?? "data/processed/Lua\314\243\314\202t-31-2024-QH15.txt"
?? "data/processed/Lua\314\243\314\202t-43-2024-QH15.stats.json"
?? "data/processed/Lua\314\243\314\202t-43-2024-QH15.txt"
?? "data/processed/Lua\314\243\314\202t-45-2013-QH13.stats.json"
?? "data/processed/Lua\314\243\314\202t-45-2013-QH13.txt"
?? "data/processed/Nghi\314\243-quye\314\202\314\201t-254-2025-QH15.stats.json"
?? "data/processed/Nghi\314\243-quye\314\202\314\201t-254-2025-QH15.txt"
?? "data/raw_doc/Lua\314\243\314\202t-31-2024-QH15.docx"
?? "data/raw_doc/Lua\314\243\314\202t-43-2024-QH15.docx"
?? "data/raw_doc/Lua\314\243\314\202t-45-2013-QH13.docx"
?? "data/raw_doc/Nghi\314\243-quye\314\202\314\201t-254-2025-QH15.docx"


In [19]:
!git fetch origin
!git status -sb

fatal: could not read Username for 'https://github.com': No such device or address
## main...origin/main
?? "data/processed/Lua\314\243\314\202t-31-2024-QH15.stats.json"
?? "data/processed/Lua\314\243\314\202t-31-2024-QH15.txt"
?? "data/processed/Lua\314\243\314\202t-43-2024-QH15.stats.json"
?? "data/processed/Lua\314\243\314\202t-43-2024-QH15.txt"
?? "data/processed/Lua\314\243\314\202t-45-2013-QH13.stats.json"
?? "data/processed/Lua\314\243\314\202t-45-2013-QH13.txt"
?? "data/processed/Nghi\314\243-quye\314\202\314\201t-254-2025-QH15.stats.json"
?? "data/processed/Nghi\314\243-quye\314\202\314\201t-254-2025-QH15.txt"
?? "data/raw_doc/Lua\314\243\314\202t-31-2024-QH15.docx"
?? "data/raw_doc/Lua\314\243\314\202t-43-2024-QH15.docx"
?? "data/raw_doc/Lua\314\243\314\202t-45-2013-QH13.docx"
?? "data/raw_doc/Nghi\314\243-quye\314\202\314\201t-254-2025-QH15.docx"


## Cài dependency và kiểm tra T4

In [ ]:
!python -m pip install -q -r requirements/retrieval.txt

In [14]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > T4 GPU'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

Sun Oct  4 04:51:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import faiss
import sentence_transformers
print('FAISS:', faiss.__version__)
print('Sentence Transformers:', sentence_transformers.__version__)

## Validate đầu vào M2

In [ ]:
import json

CHUNKS_PATH = PROJECT_ROOT / 'data/processed/chunks.json'
REPORT_PATH = PROJECT_ROOT / 'data/processed/parsing_report.json'
chunks = json.loads(CHUNKS_PATH.read_text(encoding='utf-8'))
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
warnings = [w for item in report for w in item.get('warnings', [])]
empty_chunks = [item.get('chunk_id') for item in chunks if not item.get('text', '').strip()]
unique_ids = len(chunks) == len({item['chunk_id'] for item in chunks})
print('Số chunks:', len(chunks))
print('Parsing warnings:', len(warnings))
print('Chunk rỗng:', len(empty_chunks))
print('Chunk ID duy nhất:', unique_ids)
assert len(chunks) == 921 and not warnings and not empty_chunks and unique_ids

In [ ]:
M4_OUTPUT_DIR = PROJECT_ROOT / 'indexes/faiss'
M4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print((PROJECT_ROOT / 'configs/vector.yaml').read_text(encoding='utf-8'))
print('Output:', M4_OUTPUT_DIR)

## Chạy test trước khi build

In [ ]:
!PYTHONPATH=src python -m pytest -q

## Sinh embedding trên T4 và xây FAISS index
Cell này tải `intfloat/multilingual-e5-base`, encode 921 chunks theo batch, rồi lưu index, embeddings, metadata và manifest.

In [ ]:
!python scripts/run_m4.py --rebuild --build-only --device cuda

In [ ]:
manifest = json.loads((M4_OUTPUT_DIR / 'embedding_manifest.json').read_text(encoding='utf-8'))
metadata = json.loads((M4_OUTPUT_DIR / 'vector_metadata.json').read_text(encoding='utf-8'))
required_files = ['faiss.index', 'embeddings.npy', 'vector_metadata.json', 'embedding_manifest.json']
for name in required_files:
    path = M4_OUTPUT_DIR / name
    print(f'{name}: exists={path.exists()}, size={path.stat().st_size if path.exists() else 0:,} bytes')
print('Manifest chunks:', manifest['chunk_count'])
print('Embedding dimension:', manifest['embedding_dimension'])
print('Build metrics:', manifest['build_metrics'])
assert manifest['chunk_count'] == len(metadata) == 921
assert len({item['chunk_id'] for item in metadata}) == 921

## Load lại index và chạy semantic search mẫu

In [ ]:
!python scripts/run_m4.py --device cuda --top-k 5

In [ ]:
!python scripts/run_m4.py --device cuda --top-k 5 --query 'Người dân được bồi thường như thế nào khi Nhà nước thu hồi đất ở?'

## Hoàn thành M4 khi
- Build metrics xác nhận device là `cuda`.
- FAISS có 921 vectors, metadata có 921 bản ghi và không trùng `chunk_id`.
- Index load lại thành công, checksum khớp `chunks.json`.
- Các truy vấn mẫu trả về kết quả đúng chủ đề và có metadata pháp lý.